In [1]:
import json
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.dataset import MultiTaskEmailDataset
from src.architecture import MultiTaskEmailModel

In [2]:
df = pd.read_json('data\emails.json')  

<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
C:\Users\bben2\AppData\Local\Temp\ipykernel_20112\3241957824.py:1: SyntaxWarning: invalid escape sequence '\e'
  df = pd.read_json('data\emails.json')


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df, test_df = train_test_split(df, test_size= 0.2, random_state= 42)

train_dataset = MultiTaskEmailDataset(train_df)
test_dataset = MultiTaskEmailDataset(test_df)

train_loader = DataLoader(train_dataset, shuffle=True, batch_size = 8)

test_loader = DataLoader(test_dataset, shuffle=True, batch_size = 8)

In [4]:
model = MultiTaskEmailModel()
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MultiTaskEmailModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        labels_spam = batch["label_spam"].to(device)
        labels_priority = batch["label_priority"].to(device)
        labels_intent = batch["label_intent"].to(device)

        preds = model(input_ids, attention_mask)

        loss_spam = criterion(preds["spam"], labels_spam)
        loss_priority = criterion(preds["priority"], labels_priority)
        loss_intent = criterion(preds["intent"], labels_intent)

        batch_loss = loss_spam + loss_priority + loss_intent
        total_loss += batch_loss.item()

        total_loss.backward()

        optimizer.step()

    total_spam = 0
    correct_spam = 0
    total_priority = 0
    correct_priority = 0
    total_intent = 0
    correct_intent = 0
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            labels_spam = batch["label_spam"].to(device)
            labels_priority = batch["label_priority"].to(device)
            labels_intent = batch["label_intent"].to(device)

            preds = model(input_ids, attention_mask)

            spam_pred = torch.argmax(preds['spam'], dim=-1)
            priority_pred = torch.argmax(preds['priority'], dim=-1)
            intent_pred = torch.argmax(preds['intent'], dim=-1)

            correct_spam += (spam_pred == labels_spam).sum().item()
            total_spam += len(labels_spam)
            accuracy_spam = accuracy_score(spam_pred, labels_spam)
            f1_spam = f1_score(spam_pred, labels_spam)

            correct_priority += (priority_pred == labels_priority).sum().item()
            total_priority += len(labels_priority)
            accuracy_priority = accuracy_score(priority_pred, labels_priority)
            f1_priority = f1_score(priority_pred, labels_priority, average="macro" or "weighted")


            correct_intent += (intent_pred == labels_intent).sum().item()
            total_intent += len(labels_intent)
            accuracy_intent = accuracy_score(intent_pred, labels_intent)
            f1_intent = f1_score(intent_pred, labels_intent, average="macro" or "weighted")
    
    print(f"Train loss {total_loss}, /n accuracy_spam {accuracy_spam}, f1_spam {f1_spam} /n accuracy_priority {accuracy_priority} f1_priority {f1_priority}, /n accuracy_intent {accuracy_intent} f1_intent{f1_intent}")


KeyboardInterrupt: 